# Colab bootstrap — NNDL Saliency Project

Esegui queste celle in ordine ogni volta che apri una nuova sessione Colab.
**Regola d'oro:** salva SEMPRE i checkpoint su Drive, non solo sul disco della VM — la VM viene distrutta alla disconnessione.

## 1. Verifica GPU
Runtime > Change runtime type > GPU, poi esegui questa cella.

In [ ]:
!nvidia-smi

## 2. Monta Google Drive (per checkpoint persistenti)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

DRIVE_PROJECT_DIR = '/content/drive/MyDrive/nndl-saliency'
CHECKPOINT_DIR = f'{DRIVE_PROJECT_DIR}/checkpoints'

# Archivio persistente del dataset
DATA_ARCHIVE = f'{DRIVE_PROJECT_DIR}/salicon.tar'

# Dataset veloce usato dalla VM Colab
LOCAL_DATA_DIR = '/content/data_local'

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print('Checkpoint dir:', CHECKPOINT_DIR)
print('Dataset archive:', DATA_ARCHIVE)
print('Local dataset:', LOCAL_DATA_DIR)

## 3. Clona/aggiorna il repository
La branch scelta viene aggiornata da GitHub con `pull --ff-only`; se il checkout locale contiene modifiche incompatibili, la cella si ferma. Prima dei run finali usare la branch/commit congelati.

In [ ]:
from pathlib import Path
import subprocess

REPO_URL = 'https://github.com/markbtz/Project_NN.git'
REPO_DIR = '/content/nndl-saliency'
REPO_BRANCH = 'main'  # cambiare solo per un test esplicito di una branch

if (Path(REPO_DIR) / '.git').exists():
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'switch', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only', 'origin', REPO_BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, REPO_URL, REPO_DIR], check=True)

subprocess.run(['git', '-C', REPO_DIR, 'log', '-1', '--oneline'], check=True)

%cd $REPO_DIR

## 4. Installa le dipendenze mancanti

In [ ]:
!pip install -q -r requirements-colab.txt

## 5. Credenziali Kaggle (solo la prima volta / se non già su Drive)
Carica `kaggle.json` quando richiesto (Kaggle > Settings > Create New Token).

In [ ]:
import os
kaggle_dir = os.path.expanduser('~/.kaggle')
os.makedirs(kaggle_dir, exist_ok=True)
kaggle_json_drive = '/content/drive/MyDrive/nndl-saliency/kaggle.json'
if os.path.exists(kaggle_json_drive):
    !cp "$kaggle_json_drive" ~/.kaggle/kaggle.json
else:
    from google.colab import files
    uploaded = files.upload()  # carica kaggle.json
    !mv kaggle.json ~/.kaggle/kaggle.json
    !cp ~/.kaggle/kaggle.json "$kaggle_json_drive"  # salva su Drive per la prossima volta
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle datasets list -s salicon

## 6. Prepara SALICON su Drive (solo la prima volta)

Questa sezione scarica SALICON direttamente sul disco locale veloce di Colab, esegue l'audit, crea un unico `salicon.tar` e lo salva su Google Drive. Se `salicon.tar` esiste già, non riscarica nulla.

In [ ]:
import os
import shutil
import subprocess

if os.path.exists(DATA_ARCHIVE):
    print('Archivio SALICON già presente su Drive:')
    print(DATA_ARCHIVE)
else:
    print('Archivio non presente: preparo SALICON per la prima volta.')

    # Rimuove solo un'eventuale copia locale incompleta della VM Colab.
    if os.path.exists(LOCAL_DATA_DIR):
        shutil.rmtree(LOCAL_DATA_DIR)
    os.makedirs(LOCAL_DATA_DIR, exist_ok=True)

    print('\n1/4 - Download da Kaggle sul disco locale...')
    subprocess.run([
        'python', 'scripts/download_salicon.py',
        '--output_dir', LOCAL_DATA_DIR
    ], check=True)

    print('\n2/4 - Audit del dataset...')
    subprocess.run([
        'python', 'scripts/audit_dataset.py',
        '--data_dir', LOCAL_DATA_DIR
    ], check=True)

    LOCAL_ARCHIVE = '/content/salicon.tar'
    if os.path.exists(LOCAL_ARCHIVE):
        os.remove(LOCAL_ARCHIVE)

    print('\n3/4 - Creazione di salicon.tar...')
    subprocess.run([
        'tar', '-cf', LOCAL_ARCHIVE,
        '-C', LOCAL_DATA_DIR, '.'
    ], check=True)

    print('\n4/4 - Copia del singolo archivio su Google Drive...')
    shutil.copy2(LOCAL_ARCHIVE, DATA_ARCHIVE)
    os.remove(LOCAL_ARCHIVE)

    print('\nFatto. Archivio persistente salvato in:')
    print(DATA_ARCHIVE)

In [ ]:
# Verifica che l'archivio persistente esista su Drive
if os.path.exists(DATA_ARCHIVE):
    size_gb = os.path.getsize(DATA_ARCHIVE) / (1024**3)
    print(f'OK: {DATA_ARCHIVE} ({size_gb:.2f} GB)')
else:
    print('ATTENZIONE: salicon.tar non è ancora presente su Drive.')

## 6bis. Prepara il dataset locale (a ogni nuova sessione)

Nelle sessioni successive NON riscaricare SALICON da Kaggle. Questa cella copia un solo archivio da Drive e lo estrae sul disco locale veloce della VM.

In [ ]:
import os
import shutil
import subprocess
import time

LOCAL_ARCHIVE = '/content/salicon.tar'

def fix_images_layout():
    """Corregge automaticamente il vecchio layout images/images/{train,val,test}."""
    images_dir = os.path.join(LOCAL_DATA_DIR, 'images')
    nested_images_dir = os.path.join(images_dir, 'images')

    if not os.path.isdir(nested_images_dir):
        return

    print('Rilevato layout images/images: correzione automatica...')

    for split in ['train', 'val', 'test']:
        src = os.path.join(nested_images_dir, split)
        dst = os.path.join(images_dir, split)

        if os.path.isdir(src) and not os.path.exists(dst):
            shutil.move(src, dst)

    if os.path.isdir(nested_images_dir) and not os.listdir(nested_images_dir):
        os.rmdir(nested_images_dir)

    print('Layout immagini corretto.')

if os.path.exists(LOCAL_DATA_DIR):
    fix_images_layout()
    print(f'{LOCAL_DATA_DIR} già presente. Salto preparazione.')
else:
    if not os.path.exists(DATA_ARCHIVE):
        raise FileNotFoundError(
            f'Archivio non trovato su Drive: {DATA_ARCHIVE}\n'
            'Esegui prima la sezione 6.'
        )

    t0 = time.time()

    print('Copio salicon.tar da Drive al disco locale...')
    shutil.copy2(DATA_ARCHIVE, LOCAL_ARCHIVE)

    print('Estraggo SALICON...')
    os.makedirs(LOCAL_DATA_DIR, exist_ok=True)
    subprocess.run([
        'tar', '-xf', LOCAL_ARCHIVE,
        '-C', LOCAL_DATA_DIR
    ], check=True)

    os.remove(LOCAL_ARCHIVE)

    # Compatibilità con il vecchio salicon.tar.
    fix_images_layout()

    print(f'Dataset pronto in {LOCAL_DATA_DIR} in {(time.time() - t0) / 60:.1f} minuti.')


In [ ]:
# Controllo finale della copia locale
!python scripts/audit_dataset.py --data_dir "$LOCAL_DATA_DIR"

### Routine per le sessioni future

Quando Colab assegna una nuova VM, esegui nell'ordine: **1 → 2 → 3 → 4 → 6bis → 7.1**.  
La **sezione 5** serve se occorrono le credenziali Kaggle; la **sezione 6** serve normalmente una sola volta, finché `salicon.tar` rimane su Drive.


## 7. Esperimenti (script del repository)

Il notebook orchestra test, training, evaluation e figure senza duplicare
l'implementazione degli script. Usare `tuning` fino al freeze del protocollo;
`internal_test` resta chiuso.

### 7.1 Preflight

In [ ]:
%cd $REPO_DIR
!python -m pytest -q

### 7.2 Run configuration

Tutti i flag sono centralizzati qui e sono `False` di default.
Per i training lunghi attivare un solo modello alla volta.

In [ ]:
MODELS = ["B0", "B1", "M1", "M1-L", "G", "M2"]

CHECKPOINT_FILES = {
    "B0": "B0_center_map.pt",
    "B1": "B1_best.pt",
    "M1": "M1_best.pt",
    "M1-L": "M1-L_best.pt",
    "G": "G_best.pt",
    "M2": "M2_best.pt",
}

RUN_TRAINING = {model: False for model in MODELS}
RUN_TUNING_EVAL = {model: False for model in MODELS}

RUN_TUNING_COMPARISONS = False
RUN_TUNING_BOOTSTRAP = False
RUN_QUALITATIVE = False

# Lasciare False fino al freeze definitivo.
RUN_FINAL_INTERNAL_TEST = False
RUN_FINAL_COMPARISONS = False
RUN_FINAL_BOOTSTRAP = False

PAIRS = [
    ("B0", "B1"),
    ("B1", "M1"),
    ("M1", "M1-L"),
    ("M1-L", "G"),
    ("M1-L", "M2"),
]

BOOTSTRAP_N_RESAMPLES = 1000
BOOTSTRAP_SEED = 42

EVALUATION_DIR = str(Path(DRIVE_PROJECT_DIR) / "evaluation")
COMPARISON_DIR = Path(EVALUATION_DIR) / "comparisons"
BOOTSTRAP_DIR = Path(EVALUATION_DIR) / "bootstrap"
FINAL_COMPARISON_DIR = Path(EVALUATION_DIR) / "internal_test_comparisons"
FINAL_BOOTSTRAP_DIR = Path(EVALUATION_DIR) / "internal_test_bootstrap"

#### Utility di orchestrazione

Queste utility comuni gestiscono soltanto il lancio di training ed evaluation. La logica effettiva resta in `scripts/train.py` e `scripts/evaluate.py`.

In [ ]:
import subprocess

def checkpoint_path(experiment):
    return Path(CHECKPOINT_DIR) / CHECKPOINT_FILES[experiment]


def run_training(experiment):
    cmd = [
        "python", "scripts/train.py",
        "--experiment", experiment,
        "--data_dir", LOCAL_DATA_DIR,
        "--checkpoint_dir", CHECKPOINT_DIR,
    ]

    if experiment == "G":
        base = checkpoint_path("M1-L")
        prior = checkpoint_path("B0")

        if not base.is_file():
            raise FileNotFoundError(f"Checkpoint base M1-L non trovato: {base}")
        if not prior.is_file():
            raise FileNotFoundError(f"Checkpoint B0 non trovato: {prior}")

        cmd += [
            "--base_checkpoint", str(base),
            "--center_prior_checkpoint", str(prior),
        ]

    subprocess.run(cmd, cwd=REPO_DIR, check=True)


def run_evaluation(experiment, split="tuning", final=False):
    ckpt = checkpoint_path(experiment)

    if not ckpt.is_file():
        raise FileNotFoundError(f"Checkpoint {experiment} non trovato: {ckpt}")

    cmd = [
        "python", "scripts/evaluate.py",
        "--experiment", experiment,
        "--checkpoint_path", str(ckpt),
        "--split", split,
        "--data_dir", LOCAL_DATA_DIR,
        "--results_dir", EVALUATION_DIR,
    ]

    if final:
        cmd.append("--final_evaluation")

    subprocess.run(cmd, cwd=REPO_DIR, check=True)

### 7.3 Training

I run restano separati per non concatenare più training lunghi nella stessa
esecuzione Colab.

#### 7.3.1 B0 — Center Prior

Baseline non neurale calcolata sulle density map del training set.

In [ ]:
if RUN_TRAINING["B0"]:
    run_training("B0")
else:
    print("B0 non eseguito.")

#### 7.3.2 B1 — Baseline neurale

ResNet18 pretrained + decoder semplice, loss MSE.

In [ ]:
if RUN_TRAINING["B1"]:
    run_training("B1")
else:
    print("B1 non eseguito.")

#### 7.3.3 M1 — Multi-scale

Estende B1 con feature multi-scala mantenendo la loss MSE.

In [ ]:
if RUN_TRAINING["M1"]:
    run_training("M1")
else:
    print("M1 non eseguito.")

#### 7.3.4 M1-L — Multi-scale + nuova loss

Stessa architettura di M1 con loss CC-loss + KLD.

In [ ]:
if RUN_TRAINING["M1-L"]:
    run_training("M1-L")
else:
    print("M1-L non eseguito.")

#### 7.3.5 G — Adaptive Center Prior

Estende M1-L con Adaptive Center Prior; richiede M1-L_best.pt e B0_center_map.pt.

In [ ]:
if RUN_TRAINING["G"]:
    run_training("G")
else:
    print("G non eseguito.")

#### 7.3.6 M2 — Hierarchical Transformer

PVTv2-B1 + `MultiScaleDecoder`, con target `density_map_prob` e loss
CC+KLD 0.5/0.5. Non richiede checkpoint base di altri modelli.

> Il nome del best checkpoint prodotto da `scripts/train.py` è `M2_best.pt`.

In [ ]:
if RUN_TRAINING["M2"]:
    run_training("M2")
else:
    print("M2 non eseguito.")

### 7.4 Training curves

Visualizza i grafici salvati da `scripts/train.py`. B0 non compare perché
non usa backpropagation.

In [ ]:
from IPython.display import Image, display

for experiment in ("B1", "M1", "M1-L", "G", "M2"):
    path = Path(CHECKPOINT_DIR) / f"{experiment}_training_loss.png"
    print(f"\n{experiment} — Training vs Validation Loss")
    if path.exists():
        display(Image(filename=str(path)))
    else:
        print(f"Grafico non disponibile: {path}")

### 7.5 Evaluation su tuning

Esegue `scripts/evaluate.py` sui checkpoint selezionati e produce CSV
per-image e summary JSON con CC, SIM e KLD.

In [ ]:
for experiment in MODELS:
    if RUN_TUNING_EVAL[experiment]:
        print(f"\nEvaluation tuning: {experiment}")
        run_evaluation(experiment)
    else:
        print(f"{experiment}: evaluation tuning non eseguita.")

### 7.6 Risultati sul tuning

Riepilogo assoluto, confronti paired per `image_id` e bootstrap CI.

#### 7.6.1 Metriche assolute

In [ ]:
import json
import pandas as pd

rows = []

for experiment in MODELS:
    path = Path(EVALUATION_DIR) / experiment / "tuning_summary.json"

    if not path.is_file():
        print(f"Summary mancante per {experiment}: {path}")
        continue

    with open(path, "r", encoding="utf-8") as f:
        summary = json.load(f)

    rows.append({
        "Model": experiment,
        "CC": summary["cc"],
        "SIM": summary["sim"],
        "KLD": summary["kld"],
        "N": summary["n_samples"],
    })

if rows:
    table = pd.DataFrame(rows)
    shown = table.copy()
    for metric in ("CC", "SIM", "KLD"):
        shown[metric] = shown[metric].map(lambda x: f"{x:.4f}")
    display(shown)
else:
    print("Nessun summary tuning disponibile.")

#### 7.6.2 Confronti per-image

Confronti previsti: B1−B0, M1−B1, M1-L−M1, G−M1-L, M2−M1-L.

I risultati vengono confrontati per-image allineando i CSV tramite `image_id`. Per ogni coppia vengono salvati i delta di CC, SIM e KLD e viene mostrata la variazione media sullo split considerato.

In [ ]:
import csv

def run_comparisons(split, output_dir, strict=False):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    for left, right in PAIRS:
        left_csv = Path(EVALUATION_DIR) / left / f"{split}_per_image.csv"
        right_csv = Path(EVALUATION_DIR) / right / f"{split}_per_image.csv"
        output_csv = output_dir / f"{right}_minus_{left}.csv"

        print(f"\n{right} - {left}")

        missing = [p for p in (left_csv, right_csv) if not p.is_file()]
        if missing:
            message = "Risultato mancante: " + ", ".join(map(str, missing))
            if strict:
                raise FileNotFoundError(message)
            print(message)
            continue

        subprocess.run(
            [
                "python", "scripts/compare_evaluations.py",
                str(left_csv), str(right_csv),
                "--left-name", left,
                "--right-name", right,
                "--output", str(output_csv),
            ],
            cwd=REPO_DIR,
            check=True,
        )

        with open(output_csv, "r", encoding="utf-8", newline="") as f:
            rows = list(csv.DictReader(f))

        means = {
            metric: sum(float(row[f"delta_{metric}"]) for row in rows) / len(rows)
            for metric in ("cc", "sim", "kld")
        }

        print(f"Campioni: {len(rows)}")
        print(f"Delta medio CC : {means['cc']:+.4f}")
        print(f"Delta medio SIM: {means['sim']:+.4f}")
        print(f"Delta medio KLD: {means['kld']:+.4f}")

In [ ]:
if RUN_TUNING_COMPARISONS:
    run_comparisons("tuning", COMPARISON_DIR, strict=False)
else:
    print("Confronti tuning non eseguiti.")

#### 7.6.3 Bootstrap confidence intervals

Bootstrap paired sui delta CC/SIM/KLD, con 1000 resample e seed 42.

Per ogni confronto paired viene eseguito il bootstrap sui delta per-image di CC, SIM e KLD usando i parametri definiti nella configurazione del notebook.

In [ ]:
import json

def run_bootstrap(comparison_dir, output_dir, strict=False, show_zero=False):
    comparison_dir = Path(comparison_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    for left, right in PAIRS:
        source = comparison_dir / f"{right}_minus_{left}.csv"
        output = output_dir / f"{right}_minus_{left}.json"

        print(f"\nBootstrap: {right} - {left}")

        if not source.is_file():
            message = f"Confronto mancante: {source}"
            if strict:
                raise FileNotFoundError(message)
            print(message)
            continue

        subprocess.run(
            [
                "python", "scripts/bootstrap_ci.py", str(source),
                "--metrics", "delta_cc", "delta_sim", "delta_kld",
                "--n_resamples", str(BOOTSTRAP_N_RESAMPLES),
                "--seed", str(BOOTSTRAP_SEED),
                "--output", str(output),
            ],
            cwd=REPO_DIR,
            check=True,
        )

        with open(output, "r", encoding="utf-8") as f:
            result = json.load(f)

        for metric in ("delta_cc", "delta_sim", "delta_kld"):
            item = result[metric]
            message = (
                f"{metric}: {item['observed_mean']:+.4f} "
                f"[{item['ci_lower']:+.4f}, {item['ci_upper']:+.4f}]"
            )
            if show_zero:
                message += f" | CI esclude 0: {item['significant_at_ci_level']}"
            print(message)

In [ ]:
if RUN_TUNING_BOOTSTRAP:
    run_bootstrap(COMPARISON_DIR, BOOTSTRAP_DIR, strict=False)
else:
    print("Bootstrap tuning non eseguito.")

### 7.7 Figure qualitative su tuning

La logica delle vecchie celle 7.6.1–7.6.4 è spostata in
`scripts/qualitative_figures.py`, mantenendo selezione per `image_id`,
cinque esempi, colormap `magma`, scala comune per riga, `alpha` di G e una colonna M2.

In [ ]:
from IPython.display import Image, display

FIGURE_DIR = Path(EVALUATION_DIR) / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
figure_path = FIGURE_DIR / "tuning_qualitative_comparison.png"

if RUN_QUALITATIVE:
    subprocess.run(
        [
            "python", "scripts/qualitative_figures.py",
            "--data_dir", LOCAL_DATA_DIR,
            "--checkpoint_dir", CHECKPOINT_DIR,
            "--results_dir", EVALUATION_DIR,
            "--output", str(figure_path),
        ],
        cwd=REPO_DIR,
        check=True,
    )

if figure_path.is_file():
    display(Image(filename=str(figure_path)))
else:
    print("Figura qualitativa non disponibile.")

### 7.8 Final internal-test evaluation

**BLOCCATA DI DEFAULT.** Evaluation, confronti e bootstrap restano separati
per non dover rilanciare tutta la pipeline finale dopo un'interruzione Colab.

#### 7.8.1 Evaluation su internal test

In [ ]:
if RUN_FINAL_INTERNAL_TEST:
    missing = [str(checkpoint_path(m)) for m in MODELS if not checkpoint_path(m).is_file()]
    if missing:
        raise FileNotFoundError(
            "Mancano checkpoint finali:\n" + "\n".join(missing)
        )

    print("ATTENZIONE: avvio evaluation finale su internal_test.")

    for experiment in MODELS:
        print(f"\nEvaluation finale: {experiment}")
        run_evaluation(experiment, split="internal_test", final=True)

    print("\nEvaluation finale completata.")
else:
    print("Internal test BLOCCATO.")

#### 7.8.2 Confronti per-image finali

In [ ]:
if RUN_FINAL_COMPARISONS:
    run_comparisons("internal_test", FINAL_COMPARISON_DIR, strict=True)
else:
    print("Confronti finali non eseguiti.")

#### 7.8.3 Bootstrap confidence intervals finali

In [ ]:
if RUN_FINAL_BOOTSTRAP:
    run_bootstrap(
        FINAL_COMPARISON_DIR,
        FINAL_BOOTSTRAP_DIR,
        strict=True,
        show_zero=True,
    )
else:
    print("Bootstrap finale non eseguito.")